### Replicon coocurrence project 


### imports & setup • load/parse data files


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# Base repo path
BASE = Path("/home/nacho/HDD16/Nacho/RepliCOOC")
RAW   = BASE / "data" / "raw"
PROC  = BASE / "data" / "processed"
RES   = BASE / "results"
STATS = RES / "stats"
FIGS  = RES / "figures"

# Input files
PLSDB_MOB = RAW / "plsdb_mob.csv"
PLSD_SUM  = RAW / "plsdb_summary.csv"

# Ensure output folders exist
PROC.mkdir(parents=True, exist_ok=True)
STATS.mkdir(parents=True, exist_ok=True)
FIGS.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 140)
BASE


### load/parse data files


In [ ]:
# Load as strings, keep blanks as ""
mob  = pd.read_csv(PLSDB_MOB, dtype=str).fillna("")
summ = pd.read_csv(PLSD_SUM,  dtype=str).fillna("")

# Standardize plasmid ID
mob  = mob.rename(columns={"NUCCORE_ACC": "plasmid_id"})
summ = summ.rename(columns={"NUCCORE_ACC": "plasmid_id"})

mob.shape, summ.shape


### define functions: split_unique_csv, first_nonempty


In [ ]:
def split_unique_csv(x: str):
    """Split comma-separated tokens, strip, and de-duplicate preserving order."""
    if not isinstance(x, str):
        return []
    parts = [p.strip() for p in x.split(",") if p.strip()]
    return list(dict.fromkeys(parts))

def first_nonempty(*vals):
    for v in vals:
        if isinstance(v, str) and v.strip():
            return v.strip()
    return ""

# Minimal taxonomy table (optional, useful later)
tax = summ[[c for c in [
    "plasmid_id",
    "TAXONOMY.TAXONOMY_genus",
    "TAXONOMY.TAXONOMY_species",
    "TAXONOMY.TAXONOMY_taxon_name"
] if c in summ.columns]].copy()

tax["genus"]   = tax.apply(lambda r: first_nonempty(r.get("TAXONOMY.TAXONOMY_genus","")), axis=1)
tax["species"] = tax.apply(lambda r: first_nonempty(r.get("TAXONOMY.TAXONOMY_species",""),
                                                    r.get("TAXONOMY.TAXONOMY_taxon_name","")), axis=1)
tax = tax[["plasmid_id","genus","species"]].drop_duplicates()
tax.head(3)


### load/parse data files • save outputs


In [ ]:
# Replication clusters live in mob['rep_type'] as CSV-like strings
if "rep_type" not in mob.columns:
    raise KeyError("Column 'rep_type' not found in plsdb_mob.csv")

rep_counts = mob["rep_type"].apply(split_unique_csv).apply(len)

rep_per_plasmid = pd.DataFrame({
    "plasmid_id": mob["plasmid_id"].values,
    "rep_count": rep_counts.values
})

# Optional: attach genus/species (handy for later stratified summaries)
rep_per_plasmid = rep_per_plasmid.merge(tax, on="plasmid_id", how="left")

# Save per-plasmid table
rep_per_plasmid.to_csv(STATS / "replicon_counts_per_plasmid.tsv", sep="\t", index=False)

# Distribution table
dist = rep_per_plasmid["rep_count"].value_counts().sort_index()
dist_df = dist.rename_axis("rep_count").reset_index(name="n_plasmids")
dist_df["percent"] = dist_df["n_plasmids"] / dist_df["n_plasmids"].sum() * 100.0
dist_df.to_csv(STATS / "replicon_counts_distribution.tsv", sep="\t", index=False)

# Quick numeric summary to inspect in the notebook
summary = {
    "N_plasmids": int(rep_per_plasmid.shape[0]),
    "min": int(rep_per_plasmid["rep_count"].min()),
    "q1": float(rep_per_plasmid["rep_count"].quantile(0.25)),
    "median": float(rep_per_plasmid["rep_count"].median()),
    "mean": float(rep_per_plasmid["rep_count"].mean()),
    "q3": float(rep_per_plasmid["rep_count"].quantile(0.75)),
    "p95": float(rep_per_plasmid["rep_count"].quantile(0.95)),
    "max": int(rep_per_plasmid["rep_count"].max()),
}
dist_df, summary


### imports & setup • define functions: clean_genus_name • save outputs • visualization


In [ ]:
import re
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter

def clean_genus_name(g):
    g = re.sub(r"\s*\([^)]*\)\s*$", "", str(g or "")).strip()
    g = re.sub(r"\s+", " ", g)
    return "Others" if g.lower() in {"other", "others"} else g

# ---- 1) Normalize & count (no collapsing beyond name cleanup) ----
genus_clean = X["genus"].map(clean_genus_name)

genus_counts_full = (
    genus_clean.value_counts(dropna=False)
    .rename_axis("genus")
    .reset_index(name="n_plasmids")
    .groupby("genus", as_index=False)["n_plasmids"].sum()
    .sort_values("n_plasmids", ascending=False)
    .reset_index(drop=True)
)

total_plasmids = int(genus_counts_full["n_plasmids"].sum())
print(f"Total plasmids counted: {total_plasmids:,}")

# ---- 2) Build Top-10 (exclude existing 'Others' and blank genus '') ----
mask_core = (genus_counts_full["genus"] != "Others") & (genus_counts_full["genus"] != "")
core = genus_counts_full.loc[mask_core].copy()

top_n = 10
top10 = core.nlargest(top_n, "n_plasmids").copy()
top10_sum = int(top10["n_plasmids"].sum())

# Everything not in top10 (including pre-existing 'Others' and blank '') goes to our final Others
others_count = total_plasmids - top10_sum
assert others_count >= 0

df_plot = pd.concat(
    [top10, pd.DataFrame([{"genus": "Others", "n_plasmids": others_count}])],
    ignore_index=True
)

# ---- 3) Plot (descending, 'Others' at bottom), no CI, formatted ticks ----
df_plot = df_plot.sort_values("n_plasmids", ascending=False).reset_index(drop=True)
# Move Others to bottom explicitly
others_row = df_plot[df_plot["genus"] == "Others"]
df_plot = pd.concat([df_plot[df_plot["genus"] != "Others"], others_row], ignore_index=True)

plt.figure(figsize=(9, max(4, len(df_plot) * 0.45)))
ax = sns.barplot(
    data=df_plot,
    y="genus",
    x="n_plasmids",
    color="steelblue",
    edgecolor="black",
    errorbar=None
)

ax.set_xlabel("Number of plasmids")
ax.set_ylabel("Genus")
ax.set_title("Plasmid count per genus (Top 10 + Others)")

# Format x ticks with thousands separators
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{int(x):,}"))

xmax = df_plot["n_plasmids"].max()
ax.set_xlim(0, xmax * 1.08)
offset = max(50, int(xmax * 0.008))

for i, row in df_plot.iterrows():
    ax.text(row["n_plasmids"] + offset, i, f"{row['n_plasmids']:,}",
            va="center", ha="left", fontsize=9)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()

out_path = FIGS / "plasmids_per_genus_top10_single_others_FIXED.png"
plt.savefig(out_path, dpi=300)
plt.show()

# ---- 4) Sanity checks ----
print(df_plot)
print(f"\nCheck: top10 ({top10_sum:,}) + Others ({others_count:,}) = {top10_sum + others_count:,} == Total ({total_plasmids:,})")


### imports & setup • define functions: clean_genus_name


In [ ]:
import re
import pandas as pd
import numpy as np

MIN_N_PER_GENUS = 100  # adjust as needed

def clean_genus_name(g):
    g = re.sub(r"\s*\([^)]*\)\s*$", "", str(g or "")).strip()
    g = re.sub(r"\s+", " ", g)
    return g

# If you already built rep_per_plasmid, skip to the filter.
if "rep_per_plasmid" not in globals():
    def split_unique_csv(x: str):
        parts = [t.strip() for t in str(x or "").split(",") if t.strip()]
        return list(dict.fromkeys(parts))
    mob["rep_tokens"] = mob["rep_type"].apply(split_unique_csv)
    rep_per_plasmid = pd.DataFrame({
        "plasmid_id": mob["plasmid_id"],
        "rep_count":  mob["rep_tokens"].apply(len)
    })
    tax = (summ[["NUCCORE_ACC","TAXONOMY.TAXONOMY_genus"]]
           .rename(columns={"NUCCORE_ACC":"plasmid_id","TAXONOMY.TAXONOMY_genus":"genus"}))
    tax["genus"] = tax["genus"].map(clean_genus_name)
    rep_per_plasmid = rep_per_plasmid.merge(tax, on="plasmid_id", how="left")

# keep genera with enough N
genus_sizes = rep_per_plasmid["genus"].value_counts()
keep_genera = genus_sizes[genus_sizes >= MIN_N_PER_GENUS].index
dfg = rep_per_plasmid[rep_per_plasmid["genus"].isin(keep_genera)].copy()

dfg.shape, dfg["genus"].nunique()


### imports & setup • statistics


In [ ]:
from scipy.stats import kruskal

groups = [g["rep_count"].values for _, g in dfg.groupby("genus")]
kw_stat, kw_p = kruskal(*groups)
print(f"Kruskal–Wallis H={kw_stat:.2f}, p={kw_p:.3e}  (H0: identical distributions across genera)")


### imports & setup • statistics


In [ ]:
import statsmodels.api as sm
import patsy
import numpy as np

# Reference = most abundant genus (change if you prefer)
ref = dfg["genus"].value_counts().idxmax()
dfg["_genus"] = pd.Categorical(dfg["genus"],
                               categories=[ref] + sorted([g for g in dfg["genus"].unique() if g != ref]))

y, Xmat = patsy.dmatrices("rep_count ~ C(_genus)", data=dfg, return_type="dataframe")

# Overdispersion check with Poisson
res_pois = sm.GLM(y, Xmat, family=sm.families.Poisson()).fit()
print("Poisson overdispersion (Pearson chi2/df):", round(res_pois.pearson_chi2 / res_pois.df_resid, 2))

# Negative Binomial (handles overdispersion)
res_nb = sm.GLM(y, Xmat, family=sm.families.NegativeBinomial(alpha=1.0)).fit()
print(res_nb.summary().as_text())

# Per-genus expected mean replicon count (on response scale)
coefs = res_nb.params
mu_ref = np.exp(coefs["Intercept"])
effects = []
for lvl in dfg["_genus"].cat.categories:
    term = f"C(_genus)[T.{lvl}]"
    mu = np.exp(coefs["Intercept"] + (coefs[term] if term in coefs else 0.0))
    effects.append((lvl, mu))
pd.DataFrame(effects, columns=["genus","expected_rep_count"]).sort_values("expected_rep_count", ascending=False).head(15)


### imports & setup • statistics


In [ ]:
from scipy.stats import chi2_contingency
from statsmodels.stats.proportion import proportion_confint

dfg["cointegrate"] = (dfg["rep_count"] >= 2).astype(int)
ct = pd.crosstab(dfg["genus"], dfg["cointegrate"])
chi2, p_chi2, dof, _ = chi2_contingency(ct)
print(f"Chi-square={chi2:.2f}, df={dof}, p={p_chi2:.3e}  (H0: same cointegrate rate across genera)")

# Effect size: Cramér’s V
n_tot = ct.values.sum()
cramers_v = np.sqrt(chi2 / (n_tot * (min(ct.shape) - 1)))
print(f"Cramér’s V = {cramers_v:.3f}")

# Per-genus rates with Wilson 95% CI
rates = (dfg.groupby("genus")["cointegrate"]
           .agg(n="count", k="sum")
           .assign(prop=lambda x: x.k/x.n)
           .reset_index())
ci_low, ci_high = proportion_confint(rates["k"], rates["n"], method="wilson")
rates["ci_low"], rates["ci_high"] = ci_low, ci_high
rates.sort_values("prop", ascending=False).head(12)


### imports & setup • define functions: clean_name • visualization


In [ ]:
import re
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

# === clean genus names ===
def clean_name(g):
    g = re.sub(r"\s*\([^)]*\)", "", str(g or "")).strip()
    return g

rates_plt = rates.copy()
rates_plt["genus"] = rates_plt["genus"].map(clean_name)
rates_plt = rates_plt[rates_plt["genus"] != ""].copy()

# Keep top 20 by sample size (n)
rates_plt = rates_plt.sort_values("n", ascending=False).head(20)
# Order by observed proportion
rates_plt = rates_plt.sort_values("prop", ascending=False).reset_index(drop=True)

# Build label with counts "genus (k/n)"
rates_plt["label"] = rates_plt.apply(
    lambda r: f"{r['genus']} ({int(r['k'])}/{int(r['n'])})", axis=1
)

# === plot ===
plt.figure(figsize=(9, max(5, 0.4 * len(rates_plt))))
y = range(len(rates_plt))

# CI lines
plt.hlines(y=y, xmin=rates_plt["ci_low"], xmax=rates_plt["ci_high"],
           lw=2, color="gray", alpha=0.8)
# Point estimates
plt.plot(rates_plt["prop"], y, "o", ms=6, color="tab:blue")

plt.gca().xaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{x:.0%}"))
plt.yticks(y, rates_plt["label"])
plt.xlabel("Cointegrate proportion (rep_count ≥ 2)")
plt.title("Cointegrate rates by genus (Wilson 95% CI) — Top 20 by N")
plt.grid(axis="x", linestyle=":", alpha=0.4)
plt.tight_layout()
plt.show()


### imports & setup • define functions: split_unique_csv, clean_genus_name • save outputs • visualization


In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

# --- ensure we have rep_per_plasmid with ['plasmid_id','rep_count','genus'] ---
def split_unique_csv(x: str):
    parts = [t.strip() for t in str(x or "").split(",") if t.strip()]
    return list(dict.fromkeys(parts))

def clean_genus_name(g):
    g = re.sub(r"\s*\([^)]*\)\s*$", "", str(g or "")).strip()
    g = re.sub(r"\s+", " ", g)
    return g

if "rep_per_plasmid" not in globals():
    mob["rep_tokens"] = mob["rep_type"].apply(split_unique_csv)
    rep_per_plasmid = pd.DataFrame({
        "plasmid_id": mob["plasmid_id"],
        "rep_count":  mob["rep_tokens"].apply(len)
    })
    tax = (summ[["NUCCORE_ACC","TAXONOMY.TAXONOMY_genus"]]
           .rename(columns={"NUCCORE_ACC":"plasmid_id","TAXONOMY.TAXONOMY_genus":"genus"}))
    tax["genus"] = tax["genus"].map(clean_genus_name)
    rep_per_plasmid = rep_per_plasmid.merge(tax, on="plasmid_id", how="left")

rep_per_plasmid["genus"] = rep_per_plasmid["genus"].map(clean_genus_name)
rep_per_plasmid = rep_per_plasmid[rep_per_plasmid["genus"] != ""].copy()

# --- pick top 20 genera by sample size ---
top20 = rep_per_plasmid["genus"].value_counts().head(20).index.tolist()
df20  = rep_per_plasmid[rep_per_plasmid["genus"].isin(top20)].copy()

# Consistent N (rep_count) range across all panels
N_max = int(df20["rep_count"].max())
N_levels = list(range(0,5))  # change to range(0,7) if you want to cap at 6

# Build % distribution per genus
dist = (
    df20.groupby("genus")["rep_count"]
        .value_counts()
        .rename("n")
        .reset_index()
        .set_index(["genus","rep_count"])
        .reindex(pd.MultiIndex.from_product([top20, N_levels], names=["genus","rep_count"]),
                 fill_value=0)
        .reset_index()
)

totals = dist.groupby("genus")["n"].transform("sum")
dist["percent"] = 100.0 * dist["n"] / totals
# keep order of top20 in plotting
dist["genus"] = pd.Categorical(dist["genus"], categories=top20, ordered=True)
dist = dist.sort_values(["genus","rep_count"])

# --- plot: 4×5 small multiples ---
nrows, ncols = 4, 5
fig, axes = plt.subplots(nrows, ncols, figsize=(15, 10), sharex=True, sharey=True)
axes = axes.flatten()

for ax, genus in zip(axes, top20):
    sub = dist[dist["genus"] == genus]
    ax.bar(sub["rep_count"], sub["percent"], edgecolor="black")
    ax.set_title(genus, fontsize=10)
    ax.yaxis.set_major_formatter(PercentFormatter(100))
    ax.set_ylim(0, max(20, np.ceil(dist["percent"].max()/5)*5))  # auto but tidy
    ax.grid(axis="y", linestyle=":", alpha=0.3)
    # optional: show N for this genus in corner
    N_genus = int(totals[sub.index[0]])
    ax.text(0.98, 0.90, f"n={N_genus:,}", ha="right", va="center", transform=ax.transAxes, fontsize=8)

# Empty panels if fewer than 20 (shouldn’t happen, but just in case)
for ax in axes[len(top20):]:
    ax.axis("off")

fig.add_subplot(111, frame_on=False)
plt.tick_params(labelcolor='none', top=False, bottom=False, left=False, right=False)
plt.xlabel("Number of replicons per plasmid (N)")
plt.ylabel("% of plasmids")
plt.suptitle("Replicon-count distributions — Top 20 genera by sample size", y=0.99, fontsize=14)
plt.tight_layout(rect=[0, 0, 1, 0.97])

out_path = FIGS / "replicon_count_distribution_top20_panels.png"
plt.savefig(out_path, dpi=300)
plt.show()
print(f"✅ Figure saved to: {out_path}")


### imports & setup • define functions: clean_genus_name • save outputs • visualization


In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import PercentFormatter

# === Clean genus names ===
def clean_genus_name(g):
    g = re.sub(r"\s*\([^)]*\)\s*$", "", str(g or "")).strip()
    g = re.sub(r"\s+", " ", g)
    return g

rep_per_plasmid["genus"] = rep_per_plasmid["genus"].map(clean_genus_name)
rep_per_plasmid = rep_per_plasmid[rep_per_plasmid["genus"] != ""].copy()

# === Select top 20 genera by sample size ===
top20 = rep_per_plasmid["genus"].value_counts().head(20).index.tolist()
df20 = rep_per_plasmid[rep_per_plasmid["genus"].isin(top20)].copy()

# === Build % distribution per genus (limit to 0–4 replicons) ===
N_levels = list(range(0, 5))  # only 0,1,2,3,4

dist = (
    df20.groupby("genus")["rep_count"]
        .value_counts()
        .rename("n")
        .reset_index()
        .set_index(["genus","rep_count"])
        .reindex(pd.MultiIndex.from_product([top20, N_levels], names=["genus","rep_count"]),
                 fill_value=0)
        .reset_index()
)

# Normalize to %
totals = dist.groupby("genus")["n"].transform("sum")
dist["percent"] = 100.0 * dist["n"] / totals
dist["genus"] = pd.Categorical(dist["genus"], categories=top20, ordered=True)

# === Plot ===
plt.figure(figsize=(10,6))
sns.barplot(
    data=dist,
    x="rep_count", y="percent",
    hue="genus",
    edgecolor="black",
    dodge=True
)

plt.xlabel("Number of replicons per plasmid (N)", fontsize=12)
plt.ylabel("% of plasmids", fontsize=12)
plt.title("Distribution of replicon counts (0–4) across top 20 genera", fontsize=14)
plt.gca().yaxis.set_major_formatter(PercentFormatter(100))
plt.xticks(N_levels)
plt.grid(axis="y", linestyle=":", alpha=0.3)
plt.legend(title="Genus", bbox_to_anchor=(1.02, 1), loc="upper left", ncol=1)
plt.tight_layout()

out_path = FIGS / "replicon_count_distribution_top20_grouped_0to4.png"
plt.savefig(out_path, dpi=300)
plt.show()

print(f"✅ Figure saved to: {out_path}")


### imports & setup • define functions: split_unique_csv, clean_genus_name, fmt_pval • load/parse data files • save outputs • visualization • statistics


In [ ]:
# ============================================
# Replicon count vs plasmid size (facet por género Top-20)
# excluyendo plásmidos sin replicones (rep_count = 0)
# ============================================
import re
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from pathlib import Path

# ---- rutas
BASE = Path("/home/nacho/HDD16/Nacho/RepliCOOC")
FIGS = BASE / "results" / "figures"; FIGS.mkdir(parents=True, exist_ok=True)
STATS = BASE / "results" / "stats";   STATS.mkdir(parents=True, exist_ok=True)

# ---- helpers
def split_unique_csv(x: str):
    parts = [t.strip() for t in str(x or "").split(",") if t.strip()]
    return list(dict.fromkeys(parts))  # sin duplicados, preservando orden

def clean_genus_name(g):
    g = re.sub(r"\s*\([^)]*\)\s*$", "", str(g or "")).strip()
    g = re.sub(r"\s+", " ", g)
    return g

def fmt_pval(p):
    if p is None or np.isnan(p):
        return "NA"
    if p < 1e-4:
        return f"{p:.1e}"
    return f"{p:.4f}"

# ---- construir rep_per_plasmid si no existe
if "rep_per_plasmid" not in globals():
    assert "mob" in globals(),  "Necesito `mob` cargado (plsdb_mob.csv)."
    assert "summ" in globals(), "Necesito `summ` cargado (plsd_summary.csv)."

    mob = mob.copy()
    if "plasmid_id" not in mob.columns and "NUCCORE_ACC" in mob.columns:
        mob = mob.rename(columns={"NUCCORE_ACC":"plasmid_id"})
    mob["rep_tokens"] = mob["rep_type"].apply(split_unique_csv)

    rep_per_plasmid = pd.DataFrame({
        "plasmid_id": mob["plasmid_id"],
        "rep_count":  mob["rep_tokens"].apply(len)
    })

    # detectar columna de género en summ
    genus_col = None
    for c in ["TAXONOMY.TAXONOMY_genus", "genus", "Genus"]:
        if c in summ.columns:
            genus_col = c; break
    assert genus_col is not None, "No encuentro columna de género en `summ`."

    id_col_summ = "plasmid_id" if "plasmid_id" in summ.columns else \
                  ("NUCCORE_ACC" if "NUCCORE_ACC" in summ.columns else None)
    assert id_col_summ is not None, "No encuentro columna de ID en `summ`."

    tax = (summ[[id_col_summ, genus_col]]
           .rename(columns={id_col_summ:"plasmid_id", genus_col:"genus"}))
    tax["genus"] = tax["genus"].map(clean_genus_name)

    rep_per_plasmid = rep_per_plasmid.merge(tax, on="plasmid_id", how="left")

# limpieza básica
rep_per_plasmid = rep_per_plasmid.copy()
rep_per_plasmid["genus"] = rep_per_plasmid["genus"].map(clean_genus_name)
rep_per_plasmid = rep_per_plasmid[rep_per_plasmid["genus"] != ""].copy()

# ---- añadir tamaño del plásmido desde `summ`
len_id_col = "plasmid_id" if "plasmid_id" in summ.columns else \
             ("NUCCORE_ACC" if "NUCCORE_ACC" in summ.columns else None)
assert len_id_col is not None, "No encuentro columna de ID para longitudes en `summ`."

len_df = summ[[len_id_col, "Length"]].rename(columns={len_id_col:"plasmid_id"})
len_df["Length"] = pd.to_numeric(len_df["Length"], errors="coerce")
len_df = len_df[len_df["Length"].notna() & (len_df["Length"] > 0)].copy()

df = rep_per_plasmid.merge(len_df, on="plasmid_id", how="inner")
df["Length_kb"] = df["Length"] / 1000.0

# === Filtrar solo plásmidos con ≥1 replicón ===
df = df[df["rep_count"] > 0].copy()
print(f"[info] Plásmidos con al menos 1 replicón: {len(df):,}")

# ---- Top-20 géneros por N
top20 = df["genus"].value_counts().head(20).index.tolist()
df20  = df[df["genus"].isin(top20)].copy()
df20["genus"] = pd.Categorical(df20["genus"], categories=top20, ordered=True)

print(f"[info] Plásmidos con tamaño + rep_count>0 en Top20: {len(df20):,}")

# ---- correlaciones Spearman por género (guardamos ρ y p)
rows = []
for g, sub in df20.groupby("genus"):
    if sub["rep_count"].nunique() > 1:
        rho, p = spearmanr(sub["rep_count"], sub["Length"], nan_policy="omit")
    else:
        rho, p = np.nan, np.nan
    rows.append((g, len(sub), rho, p))
corr_tab = (pd.DataFrame(rows, columns=["genus","n","spearman_rho","p_value"])
            .sort_values("n", ascending=False))
corr_tab.to_csv(STATS / "repcount_vs_length_spearman_by_genus_top20.tsv",
                sep="\t", index=False)
display(corr_tab.head(10))

# ---- facet: scatter + línea (regresión lineal) en escala log(kb)
g = sns.lmplot(
    data=df20,
    x="rep_count", y="Length_kb",
    col="genus", col_wrap=5, height=2.8, aspect=1.1,
    scatter_kws=dict(s=12, alpha=0.35, edgecolor="none"),
    line_kws=dict(linewidth=1.4, alpha=0.9),
    robust=True, ci=None, truncate=True
)
for ax in g.axes.flatten():
    ax.set_yscale("log")
    ax.grid(True, axis="y", linestyle=":", alpha=0.3)
    ax.set_xlabel("Replicons per plasmid (N)")
    ax.set_ylabel("Plasmid size (kb, log)")

# ---- anotar ρ, p y n en cada panel
rho_map = dict(zip(corr_tab["genus"].astype(str), corr_tab["spearman_rho"]))
p_map   = dict(zip(corr_tab["genus"].astype(str), corr_tab["p_value"]))
n_map   = dict(zip(corr_tab["genus"].astype(str), corr_tab["n"]))

for ax in g.axes.flatten():
    title = ax.get_title().replace("genus = ", "")
    rho   = rho_map.get(title, np.nan)
    p     = p_map.get(title, np.nan)
    n     = n_map.get(title, 0)
    rho_txt = "NA" if np.isnan(rho) else f"{rho:+.2f}"
    p_txt   = fmt_pval(p)
    ax.text(0.98, 0.05, f"ρ={rho_txt}\n p={p_txt}\n n={n}",
            ha="right", va="bottom", transform=ax.transAxes, fontsize=8,
            bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="0.8", alpha=0.9))

plt.suptitle("Replicon count vs plasmid size (kb, log scale)\nTop 20 genera by sample size (rep_count ≥ 1)",
             y=1.02, fontsize=13)
plt.tight_layout()

out_path = FIGS / "repcount_vs_length_facet_top20_min1rep.png"
plt.savefig(out_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"✅ Figure saved to: {out_path}")
print(f"✅ Correlations table saved to: {STATS/'repcount_vs_length_spearman_by_genus_top20.tsv'}")


### imports & setup


In [ ]:
import re
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix

# --- 1) Clean genus into a NEW column (do not edit X["genus"]) ---
X = X.copy()
X["genus_clean"] = (
    X["genus"].astype(str)
      .str.replace(r"\s*\([^)]*\)", "", regex=True)  # strip "(taxid)" etc.
      .str.strip()
)
X["genus_clean_lower"] = X["genus_clean"].str.lower()

# --- 2) Define ESKAPEE (case-insensitive) and filter ---
ESKAPEE = {"Enterococcus", "Staphylococcus", "Klebsiella",
           "Acinetobacter", "Pseudomonas", "Enterobacter", "Escherichia"}
ESKAPEE_lower = {g.lower() for g in ESKAPEE}

X_eskapee = X[X["genus_clean_lower"].isin(ESKAPEE_lower)].copy()

print(f"[info] Total plasmids (ESKAPEE only): {len(X_eskapee):,}")
print(f"[info] Genera present: {sorted(X_eskapee['genus_clean'].unique())}")

# --- 3) Feature prevalence settings (ESKAPEE subset) ---
MIN_PREVALENCE_ESKAPEE = 10    # less strict since dataset is smaller
REQUIRE_GENERA_ESKAPEE = 2     # feature must appear in ≥2 ESKAPEE genera

# all_reps should already exist (list of rep columns); if not, derive it:
if "all_reps" not in globals():
    # fallback: infer replicon columns as those starting with "rep_cluster_"
    all_reps = [c for c in X_eskapee.columns if str(c).startswith("rep_cluster_")]

# --- 4) Keep replicons sufficiently common within ESKAPEE ---
features_eskapee = [r for r in all_reps if r in X_eskapee.columns and X_eskapee[r].sum() >= MIN_PREVALENCE_ESKAPEE]

present_in_eskapee = {
    f: (X_eskapee.groupby("genus_clean")[f].sum() > 0).sum() for f in features_eskapee
}
features_eskapee = sorted([f for f in features_eskapee if present_in_eskapee[f] >= REQUIRE_GENERA_ESKAPEE])

print(f"[info] replicons kept in ESKAPEE subset: {len(features_eskapee)}")

# --- 5) Sparse co-occurrence and candidate pairs by support (n11) ---
M_eskapee = csr_matrix(X_eskapee[features_eskapee].values, dtype=np.uint8)
cooc_eskapee = (M_eskapee.T @ M_eskapee).astype(np.int32).toarray()
np.fill_diagonal(cooc_eskapee, 0)

MIN_SUPPORT_ESKAPEE = 5  # lower threshold since smaller dataset
cand_i, cand_j = np.where(np.triu(cooc_eskapee, k=1) >= MIN_SUPPORT_ESKAPEE)
n11 = cooc_eskapee[cand_i, cand_j]

print(f"[info] candidate co-occurrence pairs in ESKAPEE (n11 ≥ {MIN_SUPPORT_ESKAPEE}): {len(n11):,}")

# (Optional) build a tidy candidates DataFrame for later Fisher/CMH steps
idx2feat = np.array(features_eskapee)
candidates_eskapee = pd.DataFrame({
    "A": idx2feat[cand_i],
    "B": idx2feat[cand_j],
    "n11": n11
})
candidates_eskapee.head()


### imports & setup • define functions: fisher_or, cmh_by_genus • load/parse data files • save outputs • statistics


In [ ]:
# =========================
# ESKAPEE co-occurrence: Fisher + CMH (genus-stratified)
# =========================
import re
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.contingency_tables import StratifiedTable
from scipy.sparse import csr_matrix

# ---- config / paths
BASE  = Path("/home/nacho/HDD16/Nacho/RepliCOOC")
RES   = BASE / "results"
STATS = RES / "stats"; STATS.mkdir(parents=True, exist_ok=True)

MIN_PREVALENCE_ESKAPEE = 10    # replicon must occur in >= this many ESKAPEE plasmids
REQUIRE_GENERA_ESKAPEE = 2     # replicon present in >= this many ESKAPEE genera
MIN_SUPPORT_ESKAPEE    = 5     # require >= this many co-occurrences (n11) for a pair

# ---- 0) Clean genus into NEW columns (do not touch X["genus"])
X = X.copy()
X["genus_clean"] = (
    X["genus"].astype(str)
      .str.replace(r"\s*\([^)]*\)", "", regex=True)  # strip "(taxid)" etc.
      .str.strip()
)
X["genus_clean_lower"] = X["genus_clean"].str.lower()

# ---- 1) ESKAPEE subset (case-insensitive)
ESKAPEE = {"Enterococcus", "Staphylococcus", "Klebsiella",
           "Acinetobacter", "Pseudomonas", "Enterobacter", "Escherichia"}
ESKAPEE_lower = {g.lower() for g in ESKAPEE}

X_eskapee = X[X["genus_clean_lower"].isin(ESKAPEE_lower)].copy()

print(f"[info] Total plasmids (ESKAPEE only): {len(X_eskapee):,}")
print(f"[info] Genera present: {sorted(X_eskapee['genus_clean'].unique())}")

# ---- 2) Build / check replicon feature list within ESKAPEE
# If you already have 'all_reps', we use it; otherwise infer (columns starting with 'rep_cluster_' or known names)
if "all_reps" not in globals():
    # Try to infer commonly named replicon columns (rep_cluster_*, Inc*, Col*, etc.)
    all_reps = [c for c in X_eskapee.columns
                if str(c).startswith("rep_cluster_")
                or str(c).startswith(("Inc", "Col", "rep_"))]

# Keep replicons with enough prevalence and present in >= required ESKAPEE genera
features_eskapee = [r for r in all_reps if r in X_eskapee.columns and X_eskapee[r].sum() >= MIN_PREVALENCE_ESKAPEE]
present_in_eskapee = {f: (X_eskapee.groupby("genus_clean")[f].sum() > 0).sum() for f in features_eskapee}
features_eskapee = sorted([f for f in features_eskapee if present_in_eskapee[f] >= REQUIRE_GENERA_ESKAPEE])

print(f"[info] replicons kept in ESKAPEE subset: {len(features_eskapee)}")

# ---- 3) Sparse co-occurrence and candidate pairs by support (n11)
M_eskapee = csr_matrix(X_eskapee[features_eskapee].values, dtype=np.uint8)
cooc_eskapee = (M_eskapee.T @ M_eskapee).astype(np.int32).toarray()
np.fill_diagonal(cooc_eskapee, 0)

cand_i, cand_j = np.where(np.triu(cooc_eskapee, k=1) >= MIN_SUPPORT_ESKAPEE)
n11 = cooc_eskapee[cand_i, cand_j]
idx2feat = np.array(features_eskapee)
candidates_eskapee = pd.DataFrame({"A": idx2feat[cand_i], "B": idx2feat[cand_j], "n11": n11})
print(f"[info] candidate co-occurrence pairs in ESKAPEE (n11 ≥ {MIN_SUPPORT_ESKAPEE}): {len(candidates_eskapee):,}")
display(candidates_eskapee.head())

# ---- 4) Fisher on candidates (pooled, Haldane–Anscombe OR when needed)
def fisher_or(n11, n10, n01, n00):
    table = np.array([[n11, n10],[n01, n00]], dtype=int)
    if (table == 0).any():
        OR = ((n11+0.5)*(n00+0.5))/((n10+0.5)*(n01+0.5))  # HA correction for OR only
        p  = fisher_exact(table, alternative="two-sided")[1]
    else:
        OR, p = fisher_exact(table, alternative="two-sided")
    return OR, p

# Build marginals fast
M_bool = X_eskapee[features_eskapee].astype(bool).values
N = M_bool.shape[0]
col_sum = M_bool.sum(axis=0).astype(int)
feat_index = {f:i for i,f in enumerate(features_eskapee)}

cand = candidates_eskapee.copy()
cand["i"] = cand["A"].map(feat_index)
cand["j"] = cand["B"].map(feat_index)
cand["n1."] = cand["i"].map(lambda k: col_sum[k])
cand["n.1"] = cand["j"].map(lambda k: col_sum[k])
cand["N"]   = N
cand["n10"] = cand["n1."] - cand["n11"]
cand["n01"] = cand["n.1"] - cand["n11"]
cand["n00"] = cand["N"]   - (cand["n11"] + cand["n10"] + cand["n01"])

ORs, Ps = [], []
for r in cand.itertuples(index=False):
    OR, p = fisher_or(r.n11, r.n10, r.n01, r.n00)
    ORs.append(OR); Ps.append(p)

fish = cand[["A","B","n11","n10","n01","n00","N"]].copy()
fish["OR"]     = np.array(ORs, dtype=float)
fish["log2OR"] = np.log2(fish["OR"])
fish["p"]      = np.array(Ps, dtype=float)
fish["FDR"]    = multipletests(fish["p"].values, method="fdr_bh")[1]
fish["direction"] = np.where(fish["OR"]>1, "positive",
                             np.where(fish["OR"]<1, "negative", "neutral"))

fish_out = STATS / "eskapee_pairwise_fisher.tsv"
fish.to_csv(fish_out, sep="\t", index=False)
print(f"[Fisher] tested {len(fish):,} ESKAPEE pairs; FDR<0.05: {(fish['FDR']<0.05).sum():,}")
display(fish.sort_values(["FDR","n11"], ascending=[True, False]).head(10))
print(f"Saved: {fish_out}")

# ---- 5) CMH stratified by genus_clean (within ESKAPEE) — shape fix (2x2xK)
def cmh_by_genus(df, A, B, min_support=MIN_SUPPORT_ESKAPEE):
    mats, support = [], 0
    for g, sub in df.groupby("genus_clean"):
        xA = sub[A].astype(bool).values
        xB = sub[B].astype(bool).values
        n11 = int((xA & xB).sum()); n10 = int((xA & ~xB).sum())
        n01 = int((~xA & xB).sum()); n00 = int((~xA & ~xB).sum())
        mats.append(np.array([[n11, n10],[n01, n00]], dtype=int))
        support += n11
    if len(mats) < 2 or support < min_support:
        return None
    tables = np.stack(mats, axis=2).astype(float)  # (2,2,K)
    st = StratifiedTable(tables)
    return {
        "K_strata": tables.shape[2],
        "support":  support,
        "OR_CMH":   float(st.oddsratio_pooled),
        "log2OR_CMH": float(np.log2(st.oddsratio_pooled)),
        "p_CMH":    float(st.test_null_odds().pvalue)
    }

rows = []
for r in cand.itertuples(index=False):
    out = cmh_by_genus(X_eskapee, r.A, r.B, min_support=MIN_SUPPORT_ESKAPEE)
    if out is not None:
        rows.append((r.A, r.B, out["K_strata"], out["support"], out["OR_CMH"], out["log2OR_CMH"], out["p_CMH"]))

cmh = pd.DataFrame(rows, columns=["A","B","K_strata","support","OR_CMH","log2OR_CMH","p_CMH"])
if not cmh.empty:
    cmh["FDR_CMH"] = multipletests(cmh["p_CMH"].values, method="fdr_bh")[1]
    cmh["direction"] = np.where(cmh["OR_CMH"]>1, "positive",
                                np.where(cmh["OR_CMH"]<1, "negative", "neutral"))
else:
    cmh["FDR_CMH"] = []
    cmh["direction"] = []

cmh_out = STATS / "eskapee_pairwise_cmh_by_genus.tsv"
cmh.to_csv(cmh_out, sep="\t", index=False)
sig_cmh = cmh.query("FDR_CMH < 0.05").copy()
sig_out = STATS / "eskapee_pairwise_cmh_by_genus_significant.tsv"
sig_cmh.to_csv(sig_out, sep="\t", index=False)

print(f"[CMH] computed {len(cmh):,} pairs (K≥2, support≥{MIN_SUPPORT_ESKAPEE}); FDR<0.05: {len(sig_cmh):,}")
if not sig_cmh.empty:
    top_pos = sig_cmh.sort_values(["log2OR_CMH","support"], ascending=[False, False]).head(10)
    top_neg = sig_cmh.sort_values(["log2OR_CMH","support"], ascending=[True,  False]).head(10)
    print("\nTop positive CMH associations:")
    display(top_pos[["A","B","K_strata","support","OR_CMH","log2OR_CMH","FDR_CMH"]])
    print("\nTop negative CMH associations (putative exclusion):")
    display(top_neg[["A","B","K_strata","support","OR_CMH","log2OR_CMH","FDR_CMH"]])
else:
    print("No CMH-significant pairs at current thresholds.")

print(f"Saved: {cmh_out}")
print(f"Saved: {sig_out}")


### imports & setup • define functions: per_genus_breakdown, cmh_summary • load/parse data files • save outputs • statistics


In [ ]:
# ============================================
# Classify pairs by per-genus consistency / specificity / sign changes
# Requires: X_eskapee (with 'genus_clean'), candidates_eskapee (A,B), and optionally 'cmh' table
# ============================================
import numpy as np
import pandas as pd
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.contingency_tables import StratifiedTable
from pathlib import Path

# ---- parameters (tune as needed)
ALPHA_GENUS      = 0.05     # per-genus significance (we'll BH-correct *within a pair* across genera)
MIN_INFORM_GENUS = 1        # need at least this many genera with an informative table (nonzero total)
MIN_SUPPORT_PAIR = 5        # overall n11 across genera (already used earlier; keep for safety)

RES   = Path("/home/nacho/HDD16/Nacho/RepliCOOC/results")
STATS = RES / "stats"; STATS.mkdir(parents=True, exist_ok=True)

# Utility: Haldane–Anscombe OR + two-sided Fisher p

def per_genus_breakdown(df, A, B):
    rows = []
    for g, sub in df.groupby("genus_clean"):
        xA = sub[A].astype(bool).values
        xB = sub[B].astype(bool).values
        n11 = int((xA & xB).sum())
        n10 = int((xA & ~xB).sum())
        n01 = int((~xA & xB).sum())
        n00 = int((~xA & ~xB).sum())
        N   = n11 + n10 + n01 + n00
        if N == 0:
            continue
        OR, p = fisher_or(n11, n10, n01, n00)
        rows.append((g, n11, n10, n01, n00, N, OR, np.log2(OR), p))
    out = pd.DataFrame(rows, columns=["genus","n11","n10","n01","n00","N","OR","log2OR","p"])
    return out

def cmh_summary(df, A, B):
    mats, support = [], 0
    for g, sub in df.groupby("genus_clean"):
        xA = sub[A].astype(bool).values
        xB = sub[B].astype(bool).values
        n11 = int((xA & xB).sum()); n10 = int((xA & ~xB).sum())
        n01 = int((~xA & xB).sum()); n00 = int((~xA & ~xB).sum())
        mats.append(np.array([[n11, n10],[n01, n00]], dtype=int))
        support += n11
    inform = sum((m.sum() > 0) for m in mats)
    if inform < 2 or support < MIN_SUPPORT_PAIR:
        return None
    tables = np.stack(mats, axis=2).astype(float)  # (2,2,K)
    st = StratifiedTable(tables)
    return {
        "K_strata": tables.shape[2],
        "support":  support,
        "OR_CMH":   float(st.oddsratio_pooled),
        "log2OR_CMH": float(np.log2(st.oddsratio_pooled)),
        "p_CMH":    float(st.test_null_odds().pvalue)
    }

# ---- main loop over pairs
records = []
for r in candidates_eskapee.itertuples(index=False):
    A, B, support_pair = r.A, r.B, int(r.n11)
    # Per-genus table + within-pair FDR across genera
    pg = per_genus_breakdown(X_eskapee, A, B)
    if pg.empty or pg.shape[0] < MIN_INFORM_GENUS:
        continue
    # BH across the genera for THIS pair
    pg["p_adj_pair"] = multipletests(pg["p"].values, method="fdr_bh")[1]
    pg["sig"] = pg["p_adj_pair"] < ALPHA_GENUS
    pg["sign"] = np.where(pg["log2OR"]>0, "+", np.where(pg["log2OR"]<0, "-", "0"))

    # counts by sign among significant genera
    pos_sig = int(((pg["sig"]) & (pg["log2OR"]>0)).sum())
    neg_sig = int(((pg["sig"]) & (pg["log2OR"]<0)).sum())
    n_sig   = int(pg["sig"].sum())

    # attempt CMH (robust global)
    cmh = cmh_summary(X_eskapee, A, B)
    if cmh is not None:
        OR_CMH, log2OR_CMH, p_CMH = cmh["OR_CMH"], cmh["log2OR_CMH"], cmh["p_CMH"]
        K_strata, cmh_support = cmh["K_strata"], cmh["support"]
    else:
        OR_CMH, log2OR_CMH, p_CMH, K_strata, cmh_support = np.nan, np.nan, np.nan, np.nan, support_pair

    # classification
    # 1) sign-changing if both pos_sig and neg_sig >= 1
    if (pos_sig >= 1) and (neg_sig >= 1):
        category = "sign-changing"
    # 2) consistent if ≥2 significant genera and all significant signs are the same
    elif n_sig >= 2 and ((pos_sig == n_sig) or (neg_sig == n_sig)):
        category = "consistent"
    # 3) genus-specific if exactly one significant genus
    elif n_sig == 1:
        category = "genus-specific"
    else:
        category = "uncertain"

    records.append({
        "A": A, "B": B,
        "support_total_n11": support_pair,
        "per_genus_tested": pg.shape[0],
        "per_genus_sig_total": n_sig,
        "per_genus_sig_pos": pos_sig,
        "per_genus_sig_neg": neg_sig,
        "category": category,
        "OR_CMH": OR_CMH, "log2OR_CMH": log2OR_CMH, "p_CMH": p_CMH,
        "K_strata": K_strata, "cmh_support": cmh_support
    })

summary_pairs = pd.DataFrame.from_records(records)
# Attach FDR on CMH (across pairs) where available
mask_cmh = summary_pairs["p_CMH"].notna()
summary_pairs.loc[mask_cmh, "FDR_CMH"] = multipletests(summary_pairs.loc[mask_cmh, "p_CMH"].values, method="fdr_bh")[1]
summary_pairs["direction_CMH"] = np.where(summary_pairs["log2OR_CMH"]>0, "positive",
                                          np.where(summary_pairs["log2OR_CMH"]<0, "negative", "neutral"))

# Save master table
out_path = STATS / "eskapee_pairwise_genus_consistency_classified.tsv"
summary_pairs.to_csv(out_path, sep="\t", index=False)
print(f"Saved: {out_path}")

# Quick overview
print(summary_pairs["category"].value_counts().to_string())
display(summary_pairs.sort_values(
    ["category","per_genus_sig_total","cmh_support"], ascending=[True, False, False]
).head(20))


### imports & setup • define functions: _fisher_or, per_genus_effects_long • statistics


In [ ]:
# --- helper: compute per-genus log2(OR) and support for a list of (A,B) pairs
import numpy as np
import pandas as pd
from scipy.stats import fisher_exact

def _fisher_or(n11,n10,n01,n00):
    tbl = np.array([[n11,n10],[n01,n00]], dtype=int)
    if (tbl==0).any():
        OR  = ((n11+0.5)*(n00+0.5))/((n10+0.5)*(n01+0.5))
        p   = fisher_exact(tbl, alternative="two-sided")[1]
    else:
        OR, p = fisher_exact(tbl, alternative="two-sided")
    return OR, p

def per_genus_effects_long(X_eskapee, pairs_df, max_pairs=None):
    """Return long df with columns: pair_id, A, B, genus, n11,n10,n01,n00, log2OR, sig(within-pair BH)."""
    G_LIST = sorted(X_eskapee["genus_clean"].unique())
    if max_pairs is not None:
        pairs_df = pairs_df.head(max_pairs).copy()

    rows = []
    for i, r in enumerate(pairs_df.itertuples(index=False)):
        A, B = r.A, r.B
        for g, sub in X_eskapee.groupby("genus_clean"):
            xA = sub[A].astype(bool).values
            xB = sub[B].astype(bool).values
            n11 = int((xA & xB).sum()); n10 = int((xA & ~xB).sum())
            n01 = int((~xA & xB).sum()); n00 = int((~xA & ~xB).sum())
            OR, p = _fisher_or(n11,n10,n01,n00)
            rows.append((i, A, B, g, n11,n10,n01,n00, np.log2(OR)))
    long = pd.DataFrame(rows, columns=["pair_id","A","B","genus","n11","n10","n01","n00","log2OR"])
    return long


### imports & setup • define functions: per_genus_log2OR_sig • save outputs • visualization • statistics


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests

# ===== Helper =====

def per_genus_log2OR_sig(X_eskapee, pairs_df):
    """Return long df with per-genus log2(OR) and significance (BH within each pair)."""
    rows = []
    for pid, r in enumerate(pairs_df.itertuples(index=False)):
        A, B = r.A, r.B
        subrows = []
        for g, sub in X_eskapee.groupby("genus_clean"):
            xA = sub[A].astype(bool).values
            xB = sub[B].astype(bool).values
            n11 = int((xA & xB).sum()); n10 = int((xA & ~xB).sum())
            n01 = int((~xA & xB).sum()); n00 = int((~xA & ~xB).sum())
            OR, p = _fisher_or(n11,n10,n01,n00)
            subrows.append((A,B,g,np.log2(OR),p))
        tmp = pd.DataFrame(subrows, columns=["A","B","genus","log2OR","p"])
        tmp["p_adj"] = multipletests(tmp["p"].values, method="fdr_bh")[1]
        tmp["sig"] = tmp["p_adj"] < 0.05
        tmp["pair_label"] = f"{A} ~ {B}"
        rows.append(tmp)
    return pd.concat(rows, ignore_index=True)

# ===== Select pairs =====
cmh_sig = sig_cmh.copy()
pos10 = cmh_sig.sort_values("log2OR_CMH", ascending=False).head(10)
neg10 = cmh_sig.sort_values("log2OR_CMH", ascending=True).head(10)
pairs20 = pd.concat([pos10.assign(group="positive"),
                     neg10.assign(group="negative")], ignore_index=True)
pairs20["pair_label"] = pairs20["A"] + " ~ " + pairs20["B"]

# ===== Compute per-genus effects =====
long = per_genus_log2OR_sig(X_eskapee, pairs20[["A","B"]])

# set non-significant cells to NaN (white in plot)
long.loc[~long["sig"], "log2OR"] = np.nan

# pivot
mat = long.pivot_table(index="pair_label", columns="genus", values="log2OR", aggfunc="mean")

# order rows: positives first, then negatives
pairs20_sorted = pairs20.sort_values(["group","log2OR_CMH"], ascending=[True, False])
row_order = pairs20_sorted["pair_label"].tolist()
mat = mat.reindex(row_order)

# cap color range symmetrically
vmax = np.nanpercentile(np.abs(mat.values.flatten()), 95)
vmax = max(vmax, 1.0)

# ===== Plot =====
plt.figure(figsize=(10, max(5, 0.45*mat.shape[0])))
ax = sns.heatmap(mat, cmap="coolwarm", center=0, vmin=-vmax, vmax=vmax,
                 linewidths=0.3, linecolor="white", cbar_kws=dict(label="per-genus log2(OR)"),
                 mask=mat.isna())

# separator between positive and negative
n_pos = len(pos10)
ax.hlines(n_pos, *ax.get_xlim(), colors="black", linewidth=1)

# label CMH log2OR on right
cmh_map = dict(zip(pairs20["pair_label"], pairs20["log2OR_CMH"]))
for i, lab in enumerate(mat.index):
    ax.text(mat.shape[1] + 0.1, i + 0.5, f"{cmh_map.get(lab, np.nan):+.1f}",
            va="center", ha="left", fontsize=8, transform=ax.transData)

plt.title("Top 10 positive & Top 10 negative pairs (CMH)\nper-genus log2(OR) — non-significant in white")
plt.xlabel("Genus"); plt.ylabel("Replicon pair")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()

out_path = FIGS / "ESKAPEE_top20_per_genus_sigonly_heatmap.png"
plt.savefig(out_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"✅ Saved to {out_path}")


### imports & setup • save outputs • visualization • statistics


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.multitest import multipletests

# pick genus-specific pairs
genus_spec = (summary_pairs
              .query("category == 'genus-specific'")
              .sort_values(["per_genus_sig_total","cmh_support"], ascending=[False,False])
              [["A","B"]].drop_duplicates())

if genus_spec.empty:
    print("No 'genus-specific' pairs to plot.")
else:
    topN = 30
    long = per_genus_effects_long(X_eskapee, genus_spec, max_pairs=topN)
    # Within each pair, compute per-genus p-values & BH-correct to find the single significant genus
    dots = []
    for pid, sub in long.groupby("pair_id"):
        # recompute p-values for accuracy
        pvals = []
        for r in sub.itertuples(index=False):
            OR, p = _fisher_or(r.n11,r.n10,r.n01,r.n00)
            pvals.append(p)
        sub = sub.copy()
        sub["p"] = pvals
        sub["p_adj_pair"] = multipletests(sub["p"].values, method="fdr_bh")[1]
        sig = sub[sub["p_adj_pair"] < 0.05].copy()
        if sig.shape[0] == 1:  # exactly one genus
            sig["sign"] = np.where(sig["log2OR"]>0, "pos", "neg")
            dots.append(sig)
    if not dots:
        print("No pairs with exactly one significant genus among the selected.")
    else:
        D = pd.concat(dots, ignore_index=True)
        D["pair_label"] = D["A"] + " ~ " + D["B"]
        # build scatter positions
        genera = sorted(X_eskapee["genus_clean"].unique())
        pair_order = D["pair_label"].unique().tolist()
        color_map = {"pos":"#2b8cbe", "neg":"#d7301f"}

        plt.figure(figsize=(10, max(4, 0.35*len(pair_order))))
        for _, r in D.iterrows():
            x = genera.index(r["genus"]); y = pair_order.index(r["pair_label"])
            plt.scatter(x, y, s=50 + 40*np.sqrt(max(1,r["n11"])),
                        color=color_map[r["sign"]], edgecolor="black", linewidth=0.3)
        plt.yticks(range(len(pair_order)), pair_order)
        plt.xticks(range(len(genera)), genera, rotation=45, ha="right")
        plt.grid(axis="x", linestyle=":", alpha=0.3)
        plt.title("Genus-specific pairs — single significant genus (dot size ∝ n11)")
        plt.xlabel("Genus"); plt.ylabel("Replicon pair")
        plt.tight_layout()
        out_path = FIGS / "ESKAPEE_genus_specific_pairs_dotmatrix.png"
        plt.savefig(out_path, dpi=300); plt.show()
        print(f"Saved: {out_path}")


### imports & setup • visualization • statistics


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests

# --- helper: HA-corrected OR + p


### code: 


### imports & setup • define functions: clean_genus_name, split_tokens_all, mad, detect_replicon_columns_from_X, build_copy_counts_from_mob, ensure_lengths_merged… • load/parse data files • save outputs • visualization • statistics


In [ ]:
# ============================================
# Copy-aware length additivity (ESKAPEE, stratified by genus)
# EXACT total copies K in {2,3,4}; robust stats (median + MAD)
# Predicted = sum(m_r * median_single_r)
# Var(pred) = sum(m_r^2 * MAD_r^2 / n_single_r)
# ============================================
import re
import numpy as np
import pandas as pd
from pathlib import Path
from statsmodels.stats.multitest import multipletests
from scipy.stats import norm

# ---- Config
BASE  = Path("/home/nacho/HDD16/Nacho/RepliCOOC")
RES   = BASE / "results"
FIGS  = RES / "figures"; FIGS.mkdir(parents=True, exist_ok=True)
STATS = RES / "stats";   STATS.mkdir(parents=True, exist_ok=True)

K_LIST          = [2, 3, 4]   # exact total copy counts
MIN_N_SINGLE    = 10          # per (genus, replicon) singles
MIN_N_MULTI     = 5           # per (genus, multiset) multi

# ---- Helpers
def clean_genus_name(g):
    g = re.sub(r"\s*\([^)]*\)\s*$", "", str(g or "")).strip()
    g = re.sub(r"\s+", " ", g)
    return g

def split_tokens_all(x: str):
    return [t.strip() for t in str(x or "").split(",") if t.strip()]

def mad(x):
    return 1.4826 * np.median(np.abs(x - np.median(x))) if len(x) > 0 else np.nan

def detect_replicon_columns_from_X(X_df):
    # útil para limitar el universo a replicones que ya tenías en X_eskapee
    return sorted([c for c in X_df.columns if str(c).startswith(("Inc","Col","rep_cluster_","rep_"))])

def build_copy_counts_from_mob(mob_df, universe=None):
    """Devuelve:
       - copy_mat: DataFrame (index=plasmid_id, columns=replicon, int copies)
       - has_dup:  Series bool por plasmid_id si alguna copia>1
       Si 'universe' no es None, limita columnas a ese conjunto de replicones.
    """
    rows = []
    for pid, txt in zip(mob_df["plasmid_id"], mob_df["rep_type"]):
        toks = split_tokens_all(txt)
        for t in toks:
            rows.append((pid, t))
    if not rows:
        return (pd.DataFrame(index=mob_df["plasmid_id"].unique()), pd.Series(False, index=[]))
    long = pd.DataFrame(rows, columns=["plasmid_id","replicon"])
    cnt = (long.groupby(["plasmid_id","replicon"]).size()
                .rename("copies").reset_index())
    copy_mat = cnt.pivot(index="plasmid_id", columns="replicon", values="copies").fillna(0).astype(int)
    if universe is not None:
        for c in universe:
            if c not in copy_mat.columns:
                copy_mat[c] = 0
        copy_mat = copy_mat[sorted(universe)]
    has_dup = (copy_mat.max(axis=1) > 1)
    return copy_mat, has_dup

def ensure_lengths_merged(X_like, summ):
    id_col_summ = "plasmid_id" if "plasmid_id" in summ.columns else \
                  ("NUCCORE_ACC" if "NUCCORE_ACC" in summ.columns else None)
    assert id_col_summ is not None, "No encuentro columna de ID en `summ`."
    len_df = summ[[id_col_summ, "Length"]].rename(columns={id_col_summ:"plasmid_id"}).copy()
    len_df["Length"] = pd.to_numeric(len_df["Length"], errors="coerce")
    XE = X_like.merge(len_df, on="plasmid_id", how="left")
    XE = XE[XE["Length"].notna() & (XE["Length"] > 0)].copy()
    XE["genus_clean"] = XE["genus_clean"].map(clean_genus_name)
    return XE

def singles_summary_copyaware(XE, copy_mat):
    """Singles = total_copies == 1; devuelve mediana y MAD por (genus, replicon)."""
    # total copies por plasmid
    total_copies = copy_mat.sum(axis=1)
    singles_pid = total_copies[total_copies == 1].index
    if len(singles_pid) == 0:
        return pd.DataFrame(columns=["genus","replicon","n_single","median_single","mad_single"])
    # identificar qué replicón tiene copia 1
    subX = XE[XE["plasmid_id"].isin(singles_pid)][["plasmid_id","genus_clean","Length"]].copy()
    subC = copy_mat.loc[singles_pid]
    # el replicón activo por fila
    idx_max = subC.values.argmax(axis=1)
    reps = np.array(subC.columns, dtype=object)[idx_max]
    singles_long = pd.DataFrame({
        "genus": subX["genus_clean"].values,
        "replicon": reps,
        "Length": subX["Length"].values
    })
    stat = (singles_long
            .groupby(["genus","replicon"])["Length"]
            .agg(n_single="count", median_single="median", mad_single=mad)
            .reset_index())
    return stat

def combos_exact_K_copyaware(XE, copy_mat, K):
    """Agrupa por (genus, multiset con multiplicidades) para EXACTAMENTE K copias totales."""
    total_copies = copy_mat.sum(axis=1)
    pidK = total_copies[total_copies == K].index
    if len(pidK) == 0:
        return pd.DataFrame(columns=["genus","combo","n_multi","median_multi","mad_multi"])
    subX = XE[XE["plasmid_id"].isin(pidK)][["plasmid_id","genus_clean","Length"]].copy()
    subC = copy_mat.loc[pidK]
    rows = []
    for pid, row in subC.iterrows():
        nonzero = row[row > 0]
        # crear multiset ordenado lexicográficamente como tupla de (replicon, copies)
        combo = tuple((rep, int(nonzero[rep])) for rep in sorted(nonzero.index))
        g = subX.loc[subX["plasmid_id"] == pid, "genus_clean"].values[0]
        L = subX.loc[subX["plasmid_id"] == pid, "Length"].values[0]
        rows.append((g, combo, L))
    tmp = pd.DataFrame(rows, columns=["genus","combo","Length"])
    stat = (tmp.groupby(["genus","combo"])["Length"]
            .agg(n_multi="count", median_multi="median", mad_multi=mad)
            .reset_index())
    return stat

def combo_label(combo):
    # combo is tuple of (replicon, copies); format like IncFIAx2+IncFIB
    parts = []
    for rep, m in combo:
        rep2 = rep.replace("rep_cluster_","rc")
        parts.append(f"{rep2}x{m}" if m != 1 else rep2)
    return "+".join(parts)

def test_additivity_copyaware_for_K(XE, copy_mat, K, singles_stats):
    """Predicción: sum(m_r * median_single_r); Var: sum(m_r^2 * MAD_r^2 / n_r)."""
    multi_stats = combos_exact_K_copyaware(XE, copy_mat, K)
    if multi_stats.empty or singles_stats.empty:
        return pd.DataFrame()
    rows = []
    for r in multi_stats.itertuples(index=False):
        g, combo, n_multi, med_multi, mad_multi = r.genus, r.combo, r.n_multi, r.median_multi, r.mad_multi
        ok = True
        pred = 0.0
        var_pred = 0.0
        singles_detail = []
        for rep, m in combo:
            s = singles_stats[(singles_stats["genus"]==g) & (singles_stats["replicon"]==rep)]
            if s.empty or (int(s["n_single"].iloc[0]) < MIN_N_SINGLE):
                ok = False; break
            n_s  = int(s["n_single"].iloc[0])
            medS = float(s["median_single"].iloc[0])
            madS = float(s["mad_single"].iloc[0])
            pred += m * medS
            var_pred += (m**2) * (madS**2) / n_s
            singles_detail.append((rep, m, n_s, medS, madS))
        if (not ok) or (n_multi < MIN_N_MULTI):
            continue
        SE_pred = np.sqrt(var_pred)
        SE_obs  = mad_multi / np.sqrt(n_multi) if n_multi > 1 else np.nan
        SE_diff = np.sqrt(SE_pred**2 + (SE_obs**2 if not np.isnan(SE_obs) else 0.0))
        obs     = med_multi
        diff    = obs - pred
        ratio   = obs / pred if pred > 0 else np.nan
        z       = diff / SE_diff if SE_diff > 0 else np.nan
        p       = 2*(1 - norm.cdf(abs(z))) if np.isfinite(z) else np.nan

        row = {
            "K": K, "genus": g, "combo": combo, "combo_label": combo_label(combo),
            "n_multi": n_multi, "median_multi": med_multi, "mad_multi": mad_multi,
            "pred_median": pred, "obs_median": obs,
            "diff": diff, "ratio": ratio, "z": z, "p": p
        }
        rows.append(row)
    return pd.DataFrame(rows)

# ---- Preconditions
assert "X_eskapee" in globals(), "Necesito X_eskapee en memoria."
assert "summ" in globals(), "Necesito `summ` en memoria."
assert "mob" in globals(),  "Necesito `mob` en memoria (para leer rep_type con copias)."

# ---- Prepare base table with lengths + clean genus
XE = ensure_lengths_merged(X_eskapee, summ)

# ---- Build copy-count matrix from `mob` and limit to ESKAPEE plasmids
mob_use = mob[["plasmid_id","rep_type"]].dropna().copy()
rep_universe = detect_replicon_columns_from_X(XE)  # limita al universo de X
copy_mat_all, has_dup = build_copy_counts_from_mob(mob_use, universe=rep_universe)
copy_mat = copy_mat_all.reindex(XE["plasmid_id"].unique()).fillna(0).astype(int)

# ---- Singles robustos (total_copies == 1)
singles_stats = singles_summary_copyaware(XE, copy_mat)

# ---- Ejecutar por K exacto (copy-aware)
all_results = []
for K in K_LIST:
    resK = test_additivity_copyaware_for_K(XE, copy_mat, K, singles_stats)
    if not resK.empty:
        resK["FDR"] = multipletests(resK["p"].fillna(1.0).values, method="fdr_bh")[1]
        outK = STATS / f"eskapee_length_additivity_by_genus_copyaware_K{K}_median.tsv"
        resK.sort_values(["genus","FDR","n_multi"], ascending=[True, True, False]).to_csv(outK, sep="\t", index=False)
        print(f"[done] copy-aware K={K}: {len(resK):,} tests → {outK}")
        all_results.append(resK)
    else:
        print(f"[warn] copy-aware K={K}: no valid tests.")
additivity_copyaware = pd.concat(all_results, ignore_index=True) if all_results else pd.DataFrame()
if not additivity_copyaware.empty:
    out_all = STATS / "eskapee_length_additivity_by_genus_copyaware_allK_median.tsv"
    additivity_copyaware.to_csv(out_all, sep="\t", index=False)
    print(f"[done] Combined table → {out_all}")
    display(additivity_copyaware.head(20))

# ---- Scatter con etiquetas sin solape (por K)
# ---- Scatter con transparencia por significancia y etiquetas sin solape ----
if not additivity_copyaware.empty:
    import seaborn as sns
    import matplotlib.pyplot as plt
    try:
        from adjustText import adjust_text
        HAVE_ADJ = True
    except Exception:
        HAVE_ADJ = False

    for K in sorted(additivity_copyaware["K"].unique()):
        sub = additivity_copyaware[additivity_copyaware["K"] == K].copy()
        if sub.empty:
            continue

        # definir significancia
        sub["is_sig"] = sub["FDR"] < 0.05
        sig_n = sub["is_sig"].sum()
        print(f"[plot] K={K}: {sig_n}/{len(sub)} significant (FDR<0.05)")

        plt.figure(figsize=(8.2, 6.2))
        # puntos no significativos
        sns.scatterplot(
            data=sub[~sub["is_sig"]],
            x="pred_median", y="obs_median",
            hue="genus",
            alpha=0.25, edgecolor="none", legend=False
        )
        # puntos significativos
        ax = sns.scatterplot(
            data=sub[sub["is_sig"]],
            x="pred_median", y="obs_median",
            hue="genus",
            alpha=0.9, edgecolor="black", linewidth=0.3
        )

        # top 12 desviaciones absolutas entre los significativos
        topN = (sub[sub["is_sig"]]
                .reindex(sub[sub["is_sig"]]["diff"].abs().sort_values(ascending=False).index)
                .head(12))
        texts = []
        for _, r in topN.iterrows():
            txt = ax.text(
                r["pred_median"], r["obs_median"], r["combo_label"],
                fontsize=7, ha="center", va="center",
                color="black", weight="semibold"
            )
            texts.append(txt)

        # evitar solapamiento
        if HAVE_ADJ and len(texts) > 0:
            adjust_text(
                texts, ax=ax,
                arrowprops=dict(arrowstyle="-", lw=0.4, color="gray", alpha=0.6),
                expand_points=(1.05, 1.2),
                expand_text=(1.1, 1.3),
                force_points=0.5, force_text=0.7, lim=200
            )

        lim = float(np.nanmax([sub["pred_median"].max(), sub["obs_median"].max()])) * 1.05
        ax.plot([0, lim], [0, lim], linestyle="--", alpha=0.6, color="gray")

        plt.xlabel("Predicted length = Σ m_r · median(single r) (bp)")
        plt.ylabel("Observed length (exact total copies K) (bp)")
        plt.title(f"ESKAPEE — Copy-aware length additivity by genus (K={K}, medians)\nSolid = FDR<0.05, transparent = ns")
        plt.tight_layout()

        out_png = FIGS / f"ESKAPEE_length_additivity_copyaware_scatter_K{K}_sigalpha.png"
        plt.savefig(out_png, dpi=300, bbox_inches="tight")
        plt.show()
        print(f"[plot] Saved: {out_png}")
